# 02 — EDA, statistics, and SQL
Audit coverage and distributions before modelling. Never remove an outlier only because it looks unusual.

In [ ]:
%pip install -q -e .
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from agridecision.advanced.statistics import compare_market_prices
from agridecision.data.sqlite_store import query_frame, write_mandi_observations

In [ ]:
df = pd.read_csv('data/processed/mandi_prices.csv', parse_dates=['arrival_date'])
print(df.shape)
display(df.head())
display(df[['min_price', 'modal_price', 'max_price']].describe().T)

In [ ]:
coverage = df.groupby(['state', 'market']).agg(
    rows=('modal_price', 'size'),
    first_date=('arrival_date', 'min'),
    last_date=('arrival_date', 'max'),
    median_price=('modal_price', 'median'),
).sort_values('rows', ascending=False)
coverage.head(20)

In [ ]:
top_markets = coverage.head(6).reset_index()['market'].tolist()
plot_data = df[df['market'].isin(top_markets)]
plt.figure(figsize=(14, 6))
sns.lineplot(data=plot_data, x='arrival_date', y='modal_price', hue='market', estimator='median')
plt.title('Modal onion price through time')
plt.ylabel('INR per quintal')
plt.show()

In [ ]:
if len(top_markets) >= 2:
    market_a = df.loc[df['market'] == top_markets[0], 'modal_price'].to_numpy()
    market_b = df.loc[df['market'] == top_markets[1], 'modal_price'].to_numpy()
    print(compare_market_prices(market_a, market_b, paired=False))

In [ ]:
write_mandi_observations(df, 'data/processed/agridecision.db')
query_frame('data/processed/agridecision.db', '''
SELECT state, market, COUNT(*) AS rows, ROUND(AVG(modal_price), 2) AS mean_price
FROM mandi_prices
GROUP BY state, market
ORDER BY rows DESC
LIMIT 20
''')